In [65]:
import json
import pandas as pd
from sentence_transformers import SentenceTransformer
from sentence_transformers import util
import numpy as np
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
import re
from nltk.stem import WordNetLemmatizer,PorterStemmer
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Prudhvi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [66]:
with open("data/processed/chunk.json","r",encoding="utf-8") as f:
    chunks =json.load(f)

In [67]:
en_stopwords=stopwords.words("english")
ps = PorterStemmer()

In [68]:
def tokenize(text):
    text=text.lower()
    text=text.replace("u.s.","us")
    text = re.sub(r"[/\-\u2011\u2013]", " ", text)
    text_token=word_tokenize(text.lower())
    text_token_punt = [re.sub(r"[^\w\s]", "", word) for word in text_token]
    text_token_stopword = [word for word in text_token_punt
                           if word and word not in en_stopwords]

    text_token_stem = [ps.stem(word) for word in text_token_stopword]
    return text_token_stem
    

In [69]:
tokenized_chunks=[tokenize(chunk["text"]) for chunk in chunks]

In [70]:
bm25 = BM25Okapi(tokenized_chunks)

In [71]:
def bm25_search(query, k=5):
    query_token=tokenize(query)
    scores=bm25.get_scores(query_token)
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    # for i in top:
    #     print(round(scores[i], 2), chunks[i]["chunk_id"], chunks[i]["text"][:100])
    return  [(chunks[i]["chunk_id"], float(scores[i])) for i in top]

In [72]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [74]:
text=[chunk["text"] for chunk in chunks]
chunk_vecs =  np.load("data/processed/chunk_embeddings.npy")
#model.encode(text,show_progress_bar=True)

In [75]:
def Embed_search(query,k=5):
    query_vec=model.encode(query)
    query_util=util.cos_sim(query_vec,chunk_vecs)[0]  # measures how similar two embedding are, by checking whether they point in the same direction.
    top = sorted(range(len(query_util)), key=lambda i: query_util[i], reverse=True)[:k]
    # for i in top:
    #     print(round(float(query_util[i]), 2), chunks[i]["chunk_id"], chunks[i]["text"][:100])
    return [(chunks[i]["chunk_id"],float(query_util[i]))for i in top]
    

In [76]:
def hybrid_query(question,k=5):
    result_bm25=bm25_search(question,k=20)
    df_bm25=pd.DataFrame(result_bm25, columns=["chunk_id", "score"])
    df_bm25["rank"]=range(1,len(df_bm25)+1)
    result_embed=Embed_search(question,k=20)
    df_embed=pd.DataFrame(result_embed, columns=["chunk_id", "score"])
    df_embed["rank"]=range(1,len(df_embed)+1)
    final= pd.merge(df_embed,df_bm25, on="chunk_id", how="outer", suffixes=( "_embed","_bm25"))
    final["rrf"] = (1 / (60 + final["rank_bm25"])).fillna(0) + (1 / (60 + final["rank_embed"])).fillna(0) # Reciprocal Rank Fusion
    final = final.sort_values("rrf", ascending=False).reset_index(drop=True)
    return final.head(k)

### balance sheet reduction
- Top results are the 2022 chunks about reducing Treasury and MBS holdings,
  plus 2022-05-04 (when the reduction was announced) and 2025-10-29.
- BM25 alone missed 2022-05-04. Embeddings alone ranked 2021 tapering
  chunks first. Hybrid has both relevant ones in the top 5.
- Still includes one tapering chunk (2021-12-15), which is a different policy.
  

In [77]:

hybrid_query("balance sheet reduction")

,chunk_id,score_embed,rank_embed,score_bm25,rank_bm25,rrf
0,2022-11-02_p3_c2,0.404851,2.0,6.246368,1.0,0.032522
1,2022-12-14_p3_c2,0.404851,3.0,6.246368,2.0,0.032002
2,2025-10-29_p3_c1,0.356837,8.0,5.388285,3.0,0.030579
3,2021-12-15_p4_c2,0.421829,1.0,3.784495,11.0,0.030478
4,2022-05-04_p3_c1,0.375759,5.0,4.100757,10.0,0.029670


### inflation transitory
- The two strongest 2021 "transitory" chunks rank 1 and 2. They came from
  BM25 (rank 1 and 2); embeddings ranked them 11 and 13.
- The other three 2021 transitory chunks dropped out, replaced by 2024 chunks
  that both searches ranked in the middle. RRF favors chunks that are
  moderately good in both lists over chunks that are strong in only one.


In [78]:
hybrid_query("inflation transitory")

,chunk_id,score_embed,rank_embed,score_bm25,rank_bm25,rrf
0,2021-09-22_p2_c1,0.427782,11.0,4.472144,2.0,0.030214
1,2021-07-28_p2_c1,0.419378,13.0,4.714713,1.0,0.030092
2,2024-05-01_p2_c1,0.442340,5.0,1.463931,8.0,0.030090
3,2024-01-31_p2_c1,0.436827,7.0,1.492150,6.0,0.030077
4,2024-06-12_p2_c1,0.442340,6.0,1.463931,9.0,0.029644


### when did the Fed raise rates by 75 basis points
- 2022-06-15, 2022-07-27 and 2022-09-21 are in the top 5. These are three of
  the four 75bp hikes. Neither search alone found all three.
- 2022-11-02 (the fourth 75bp hike) is missing.
- The statements don't state the size of the hike, only the new range,
  so no search method can match "75" directly. This needs the LLM to compare
  ranges, or a structured table of rate decisions.


In [61]:
hybrid_query("when did the Fed raise rates by 75 basis points")

,chunk_id,score_embed,rank_embed,score_bm25,rank_bm25,rrf
0,2026-09-16_p2_c1,0.513351,1.0,5.810786,5.0,0.031778
1,2026-10-28_p5_c1,0.469051,6.0,6.627104,2.0,0.031281
2,2022-09-21_p3_c1,0.479665,4.0,2.353570,13.0,0.029324
3,2022-06-15_p3_c1,0.468132,7.0,2.313923,15.0,0.028259
4,2022-07-27_p3_c1,0.461636,12.0,2.313923,16.0,0.027047
